# Bayesian HUD Statistics

A poker HUD (Heads-Up Display) tracks opponent statistics across hundreds of hands.
The core problem is **small samples**: after 30 hands a villain's raw VPIP is a noisy
estimate of their true rate, and rare stats like 3-bet% may have only 4–5 opportunities.

This notebook develops three progressively richer Bayesian solutions:

| Use case | Question answered | Method |
|---|---|---|
| 1 — Single-stat | How much should we trust a raw sample stat? | Gaussian shrinkage toward a population prior |
| 2 — Multi-stat | Which player type are we facing? | Discrete posterior over archetypes updated jointly on all stats |
| 3 — Within-hand | Who is this *anonymous* villain? | Sequential action-by-action Bayesian updating |

**Key motivation.** A raw sample average of $\hat{\theta} = k/n$ has sampling noise
$s = \sqrt{\hat{\theta}(1-\hat{\theta})/n}$ that can dwarf the true between-player
variation $\sigma$ when $n$ is small. Bayesian shrinkage replaces $\hat{\theta}$
with an estimate that hedges toward the population mean in proportion to that noise.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from bayesian_hud.archetypes import (
    ARCHETYPES, ARCHETYPE_NAMES, STAT_NAMES, MIXTURE_WEIGHTS, get_archetype_params
)
from bayesian_hud.single_stat import (
    plot_estimation_comparison, plot_shrinkage_curves
)
from bayesian_hud.multi_stat import (
    plot_archetype_posteriors, plot_estimate_improvement,
    simulate_archetype_population
)
from bayesian_hud.decision_tree import (
    plot_posterior_evolution, plot_path_tree, trace_path
)

## Archetype Definitions

We model the player pool as a mixture of three archetypes.
Each archetype $k$ is characterised by a **Gaussian prior** over three preflop stats:

$$\theta_j \mid k \sim \mathcal{N}(\mu_{kj},\, \sigma_{kj}^2), \quad j \in \{\text{VPIP, PFR, 3B\%}\}$$

The three archetypes are:

- **Fish** (40% of pool) — loose-passive calling station. High VPIP, very low PFR and 3B%.
  Rarely folds preflop, almost never 3-bets, calls too wide on every street.
- **TAG** (45% of pool) — tight-aggressive regular. Selective preflop range, decent aggression,
  folds often to 3-bets and cbets. The most common competent player type.
- **LAG** (15% of pool) — loose-aggressive. Wide range but high aggression; 3-bets frequently,
  donk-bets and check-raises more than TAG.

The mixture weights $\pi_k$ represent each archetype's population share and serve as
the prior $P(k)$ before any hand data is observed.

In [ ]:
mu, sigma, pi = get_archetype_params()

header = f"{'Archetype':<8}  {'Weight':>6}  " + "  ".join(f"{s:>12}" for s in STAT_NAMES)
sep    = "-" * len(header)
print(header)
print(sep)
for k, name in enumerate(ARCHETYPE_NAMES):
    stat_cols = "  ".join(
        f"{mu[k,j]:.2f} ± {sigma[k,j]:.2f}" for j in range(len(STAT_NAMES))
    )
    print(f"{name:<8}  {pi[k]:>6.0%}  {stat_cols}")
print(sep)
print(f"{'Total':<8}  {pi.sum():>6.0%}")

## Use Case 1: Single-Stat Bayesian Filtering

### The shrinkage estimator

For a player with $n$ opportunities and observed rate $\hat{\theta} = k/n$,
the sampling standard deviation is:

$$\hat{s} = \sqrt{\frac{\hat{\theta}(1 - \hat{\theta})}{n}}$$

We combine this with a population prior $\mathcal{N}(\mu, \sigma^2)$ via a
**data weight** $w \in [0, 1]$:

$$w = \frac{\sigma}{\sigma + \hat{s}}, \qquad
  \hat{\theta}_B = \mu + w\,(\hat{\theta} - \mu)$$

When $n$ is small, $\hat{s} \gg \sigma$ so $w \approx 0$ and we return the prior mean.
As $n \to \infty$, $\hat{s} \to 0$ so $w \to 1$ and we return the raw average.
The **crossover point** — where the data and prior contribute equally — is:

$$n^* = \frac{\mu(1-\mu)}{\sigma^2}$$

### Why this matters for rare stats

VPIP is observed on every hand ($\text{opp. rate} = 100\%$), so $n$ grows quickly.
But 3-bet% only has an opportunity rate of ~15%, meaning after 50 total hands
we've seen roughly 7–8 3-bet opportunities — deep in the shrinkage-dominated regime.

In [ ]:
fig = plot_shrinkage_curves()
plt.show()

In [ ]:
fig = plot_estimation_comparison(total_hands=50)
plt.show()

## Use Case 2: Archetype-Based Multi-Stat Filtering

### Model

Rather than a single population prior, we maintain a **discrete posterior over archetypes**
and update it jointly on all three observed stats.
Treating $\hat{\theta}_j$ as a noisy observation of the true rate with noise $\hat{s}_j$,
the total variance under archetype $k$ for stat $j$ is:

$$v_{kj} = \sigma_{kj}^2 + \hat{s}_j^2$$

The log-likelihood of the observed vector $\hat{\boldsymbol{\theta}}$ under archetype $k$ is:

$$\log p(\hat{\boldsymbol{\theta}} \mid k) = \sum_j \log \mathcal{N}\!\left(\hat{\theta}_j;\, \mu_{kj},\, v_{kj}\right)$$

and the posterior is:

$$P(k \mid \hat{\boldsymbol{\theta}}) \propto \pi_k \cdot p(\hat{\boldsymbol{\theta}} \mid k)$$

The final estimate for stat $j$ is then the **archetype-posterior-weighted shrinkage**:

$$\hat{\theta}^{\text{arch}}_j = \sum_k P(k \mid \hat{\boldsymbol{\theta}}) \cdot \hat{\theta}^B_{kj}$$

where $\hat{\theta}^B_{kj}$ is the single-stat shrinkage estimate using archetype $k$'s prior.

### Can we separate TAG from LAG?

TAG and LAG share similar VPIP ranges but differ strongly on PFR and 3B%.
With only 50 hands — roughly 10 PFR opportunities and 7–8 3B% opportunities —
separation is difficult. With 200 hands, the posteriors sharpen considerably.
The 3×3 confusion grids below show this directly.

In [ ]:
fig = plot_archetype_posteriors(total_hands=50)
plt.suptitle(fig.texts[0].get_text() if fig.texts else '', fontsize=13)
plt.show()

In [ ]:
# With 200 hands the posteriors sharpen — TAG vs LAG confusion should drop markedly
fig = plot_archetype_posteriors(total_hands=200)
plt.show()

In [ ]:
fig = plot_estimate_improvement(total_hands=50)
plt.show()

## Use Case 3: Anonymous Player — Sequential Within-Hand Updating

When we have **no hand history** — a new player at the table or a player on a
different site — we cannot use preflop stats. Instead, we update our archetype
posterior **action by action** during the hand itself.

We model a BTN vs BB single-raised pot. At each decision node, the villain's action
has archetype-specific probabilities $P(\text{action} \mid k)$ stored in the archetype
definitions. The update rule is a simple likelihood-weighted rescaling:

$$P(k \mid a_{1:t}) \propto P(k \mid a_{1:t-1}) \cdot P(a_t \mid k)$$

The decision tree below shows all possible villain lines and the per-archetype
probabilities annotated on each edge (F = Fish, T = TAG, L = LAG).

**Reading the tree:** A villain who calls preflop, checks the flop, calls the cbet,
checks the turn, and calls the barrel follows the passive call-call-call line
stereotypically associated with Fish. A villain who donk-bets the flop or
check-raises is more likely LAG.

In [ ]:
fig = plot_path_tree()
plt.show()

In [ ]:
fig = plot_posterior_evolution()
plt.show()

## Exploring Specific Hands

`trace_path()` accepts any sequence of `(node, action)` tuples and returns the
full posterior history, making it easy to probe the model interactively.

Below we examine two contrasting lines:

- **Path 1 — Check-raise turn:** villain calls preflop, checks flop, calls cbet,
  then check-raises the turn barrel. A polarising, aggressive line.
- **Path 2 — 3-bet preflop:** villain 3-bets immediately. The hand ends there
  (we fold or 4-bet), but we still gain information about archetype.

In [ ]:
paths = {
    "Check-raise turn": [
        ("preflop",        "call"),
        ("flop_donk",      "check"),
        ("flop_vs_cbet",   "call"),
        ("turn_donk",      "check"),
        ("turn_vs_barrel", "raise"),
    ],
    "3-bet preflop": [
        ("preflop", "threbet"),
    ],
}

for path_name, seq in paths.items():
    history, final = trace_path(seq)
    print(f"\n{'='*62}")
    print(f"  {path_name}")
    print(f"{'='*62}")
    header = f"  {'Step':<4}  {'Node':<18}  {'Action':<10}  " \
             + "  ".join(f"P({n})" for n in ARCHETYPE_NAMES)
    print(header)
    print(f"  {'-'*60}")
    row_labels = [("prior", "—")] + seq
    for i, (node, action) in enumerate(row_labels):
        probs = "  ".join(f"{history[i][k]:>7.3f}" for k in range(len(ARCHETYPE_NAMES)))
        print(f"  {i:<4}  {node:<18}  {action:<10}  {probs}")

## Summary and Extensions

### What the framework achieves

Across all three use cases, Bayesian shrinkage consistently beats raw sample averages
at small sample sizes — the regime where HUD data is least reliable and most consequential.
The archetype layer adds a second dividend: by pooling information across stats, it can
identify player type even when each individual stat is too noisy to be conclusive.
The within-hand updater extends this to anonymous opponents where no history exists at all.

### Limitations

**Independence assumption.** The model treats VPIP, PFR, and 3B% as conditionally
independent given the archetype. In reality they are correlated (a player with high PFR
almost certainly has high VPIP). This underestimates posterior confidence but rarely
changes which archetype has the highest probability.

**Gaussian approximation.** True rates are bounded in $[0, 1]$, but the Gaussian prior
has unbounded support. For extreme stats (e.g. 3B% near 0) a Beta prior would be more
faithful; the Gaussian is a convenient approximation that works well away from the boundaries.

**Fixed archetype parameters.** The $\mu_{kj}$, $\sigma_{kj}$, and $\pi_k$ values are
hand-specified here. In production they should be estimated from a large labelled HUD
database via expectation-maximisation or variational inference on a Gaussian mixture model.

**Action probabilities.** The within-hand likelihoods in `archetypes.py` are heuristic.
Calibrating them from real hand histories — stratified by position and board texture —
is the most impactful improvement available.

### Natural extensions

- **Correlated stats:** replace independent Gaussians with a multivariate Gaussian
  prior $\boldsymbol{\theta} \mid k \sim \mathcal{N}(\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$.
  The covariance $\boldsymbol{\Sigma}_k$ can be estimated from data or set via elicitation.
- **Online updating:** as hands accumulate session by session, update the posterior
  incrementally rather than recomputing from scratch — the Bayesian framework supports
  this natively since today's posterior is tomorrow's prior.
- **Hierarchical model:** share strength across players at the same stake or in the same
  player pool by placing a hyperprior on $\boldsymbol{\mu}_k$.
- **Richer decision trees:** condition action probabilities on board texture (dry vs wet),
  pot odds, or villain's positional tendencies to sharpen within-hand inference.